# Phase 4: corrected-vs-baseline teacher distillation ablation

Monitors the GPU pipeline driven by `tools/run_phase4_ablation_from_cache.sh`
in tmux session `phase4`. Console output is appended to `/root/phase4_pipeline.log`
(via `tee`); this notebook pulls that same log over SSH so you can read progress
in Jupyter without attaching to tmux.

**Mission brief:** `plan.md` — MISSION BRIEF (2026-09-27). Steps: 6000-sample logit
cache ×2 (baseline + LoRA-corrected teacher), then 500-step KD training ×2 on the
same pruned candidate.

**On the GPU:** `tmux attach -t phase4` or `tail -f /root/phase4_pipeline.log`.

**In this notebook:** run cells top-to-bottom once, then re-run **Pipeline log (snapshot)** or set `LIVE=True` in the live cell for the same stream as tmux.

In [1]:
from pathlib import Path

# DigitalOcean H100 used for Phase 4 (2026-09-27)
GPU_HOST = "107.170.17.174"
GPU_USER = "root"
SSH_KEY = Path.home() / ".ssh" / "id_ed25519"

REMOTE_BONSAI = "/root/bonsai"
REMOTE_LOG = "/root/phase4_pipeline.log"
REMOTE_STATE = "/root/phase4_state.env"

print(f"monitoring {GPU_USER}@{GPU_HOST}:{REMOTE_LOG}")

monitoring root@107.170.17.174:/root/phase4_pipeline.log


In [2]:
import re
import subprocess

from IPython.display import Markdown, display


def _ssh(cmd: str, *, timeout: int = 120) -> str:
    """Run a command on the GPU host; return stdout (raises on failure)."""
    argv = [
        "ssh",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=15",
        "-i",
        str(SSH_KEY),
        f"{GPU_USER}@{GPU_HOST}",
        cmd,
    ]
    proc = subprocess.run(argv, capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or proc.stdout.strip())
    return proc.stdout


def redact_secrets(text: str) -> str:
    return re.sub(r"hf_[A-Za-z0-9]+", "hf_<redacted>", text)


def fetch_log_tail(n_lines: int = 80) -> str:
    return redact_secrets(_ssh(f"tail -n {n_lines} {REMOTE_LOG}"))


def fetch_state() -> str:
    return redact_secrets(_ssh(f"cat {REMOTE_STATE} 2>/dev/null || true"))


def fetch_gpu_snapshot() -> str:
    return _ssh(
        "nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader"
    )


def show_log(n_lines: int = 80) -> None:
    text = fetch_log_tail(n_lines)
    print(text)
    display(Markdown("```text\n" + text + "\n```"))


print("helpers loaded")

helpers loaded


In [3]:
print(fetch_state())

Streaming 6000 retraining samples (seed=11) ...
  loaded 6000 raw sample(s) from Hub
Loading tokenizer for sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora ...
Tokenizing ...
  6000 sample(s), mean length 469 tokens
Loading teacher model sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora ...

Fetching 2 files: 100%|██████████| 2/2 [01:24<00:00, 42.43s/it]

Loading weights: 100%|██████████| 531/531 [00:08<00:00, 60.63it/s]
Running teacher forward (top_k=20) on cuda:0 ...
  forward progress: 1/6000 samples (1s elapsed, ~4248s remaining)
  forward progress: 50/6000 samples (11s elapsed, ~1278s remaining)
  forward progress: 100/6000 samples (21s elapsed, ~1226s remaining)
  forward progress: 150/6000 samples (31s elapsed, ~1212s remaining)
  forward progress: 200/6000 samples (41s elapsed, ~1191s remaining)
  forward progress: 250/6000 samples (51s elapsed, ~1180s remaining)
  forward progress: 300/6000 samples (61s elapsed, ~1168s remaining)
  forward progress: 350/6000 s

In [4]:
show_log(40)

# log tail (re-run show_log to refresh)
CANDIDATE=runs/20260927T161454Z-e94ad0-build-candidate-aggressive-depth-experts/checkpoints/candidate
BASELINE_CACHE=runs/20260927T163225Z-91fcb3-cache-teacher-logits-baseline/checkpoints/baseline-topk-logits.pt


In [5]:
import json
from pathlib import Path


def parse_pipeline_status(log_text: str) -> dict[str, str | None]:
    """Best-effort read of where the pipeline is from log tail."""
    lines = log_text.splitlines()
    step = None
    forward = None
    for line in lines:
        if line.startswith("--- Step"):
            step = line.strip(" -")
        if "forward progress:" in line:
            forward = line.strip()
        if "Phase 4 ablation pipeline done" in line:
            step = "complete"
    return {"step": step, "latest_forward": forward}


def find_remote_metrics_paths() -> list[str]:
    out = _ssh(f"ls -1 {REMOTE_BONSAI}/runs/*distill*/metrics.jsonl 2>/dev/null | tail -5").strip()
    return [p for p in out.splitlines() if p]


log_tail = fetch_log_tail(120)
status = parse_pipeline_status(log_tail)
print(json.dumps(status, indent=2))
metrics = find_remote_metrics_paths()
print("metrics files:", metrics or "(none yet)")

{
  "step": "Step 2c: full lora-corrected logit cache (6000)",
  "latest_forward": "forward progress: 650/6000 samples ..."
}
metrics files: (none yet)


In [6]:
import json

import matplotlib.pyplot as plt


def load_metrics_remote(path: str) -> list[dict]:
    text = _ssh(f"cat {path}")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


paths = find_remote_metrics_paths()
if not paths:
    print("No distill metrics.jsonl yet — re-run after Step 3b starts.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    for path in paths:
        rows = load_metrics_remote(path)
        steps = [r["step"] for r in rows if "train_loss" in r]
        losses = [r["train_loss"] for r in rows if "train_loss" in r]
        label = path.split("/")[-2][:40]
        ax.plot(steps, losses, label=label)
    ax.set_xlabel("step")
    ax.set_ylabel("train_loss (KD)")
    ax.legend(fontsize=8)
    ax.set_title("Phase 4 distillation (remote metrics.jsonl)")
    plt.tight_layout()
    plt.show()

No distill metrics.jsonl yet — re-run after Step 3b starts.


In [7]:
import time

from IPython.display import clear_output

# Set LIVE=True and re-run this cell for a scrolling view (same text as tmux/log).
LIVE = False
INTERVAL_SEC = 30
ROUNDS = 20
N_LINES = 60

if not LIVE:
    show_log(N_LINES)
    print("(Set LIVE=True in this cell for auto-refresh.)")
else:
    for i in range(ROUNDS):
        clear_output(wait=True)
        print(f"refresh {i + 1}/{ROUNDS} — {INTERVAL_SEC}s interval\n")
        show_log(N_LINES)
        try:
            print("GPU:", fetch_gpu_snapshot().strip())
        except Exception as err:
            print("GPU snapshot failed:", err)
        time.sleep(INTERVAL_SEC)

(Set LIVE=True in this cell for auto-refresh.)
